# Saga Pattern for Agents (SagaLLM) | Agent Safety & Resilience

In [1]:
# Saga Pattern for Multi-Step Agent Workflows
from typing import List, Callable
from dataclasses import dataclass

In [2]:
@dataclass
class SagaStep:
    name: str
    execute: Callable[[], str]
    compensate: Callable[[], str]

class Saga:
    """Execute a series of steps with automatic compensation on failure."""

    def __init__(self, steps: List[SagaStep]):
        self.steps = steps
        self.completed: List[SagaStep] = []
        self.log: List[str] = []

    def run(self) -> dict:
        for step in self.steps:
            try:
                result = step.execute()
                self.completed.append(step)
                self.log.append(f"EXECUTED: {step.name} -> {result}")
            except Exception as e:
                self.log.append(f"FAILED: {step.name} -> {e}")
                self._compensate()
                return {"success": False, "log": self.log}
        return {"success": True, "log": self.log}

    def _compensate(self):
        """Roll back completed steps in reverse order."""
        for step in reversed(self.completed):
            try:
                result = step.compensate()
                self.log.append(f"COMPENSATED: {step.name} -> {result}")
            except Exception as e:
                self.log.append(f"COMPENSATION FAILED: {step.name} -> {e}")

In [3]:
# Simulated booking workflow
bookings = {}

def book_flight():
    bookings["flight"] = "NYC->LAX, $350"
    return "Flight booked: NYC->LAX"

def cancel_flight():
    bookings.pop("flight", None)
    return "Flight cancelled"

def book_hotel():
    bookings["hotel"] = "Hilton LAX, $200/night"
    return "Hotel booked: Hilton LAX"

def cancel_hotel():
    bookings.pop("hotel", None)
    return "Hotel cancelled"

def book_car():
    raise RuntimeError("No cars available!")  # Simulated failure

def cancel_car():
    bookings.pop("car", None)
    return "Car rental cancelled"

saga = Saga([
    SagaStep("Book Flight", book_flight, cancel_flight),
    SagaStep("Book Hotel", book_hotel, cancel_hotel),
    SagaStep("Book Car", book_car, cancel_car),
])

result = saga.run()
print(f"Success: {result['success']}")
for entry in result["log"]:
    print(f"  {entry}")
print(f"Final bookings: {bookings}")

Success: False
  EXECUTED: Book Flight -> Flight booked: NYC->LAX
  EXECUTED: Book Hotel -> Hotel booked: Hilton LAX
  FAILED: Book Car -> No cars available!
  COMPENSATED: Book Hotel -> Hotel cancelled
  COMPENSATED: Book Flight -> Flight cancelled
Final bookings: {}
